## Tools

##### Models can request to call tools that perform tasks such as fetching data from a database, searching the web or running code. Tools are pairing of:

###### 1. A Schema, including the name of the tool, a description, and/or argument definitions (often a JSON scheme)
###### 2. A function or coroutine to execute

In [2]:
from dotenv import load_dotenv
load_dotenv()

from langchain.chat_models import init_chat_model

# model = init_chat_model("groq:llama-3.3-70b-versatile")
model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("Why do parrots talk?")

print(response.content)

<think>
Okay, so I need to figure out why parrots talk. Let me start by breaking down the question. Parrots are known for mimicking human speech, but why do they do that? First, I should consider their natural behavior. Maybe talking is a way for them to communicate with other parrots? But humans aren't parrots, so why do they imitate us?

I remember reading that parrots are highly social animals. In the wild, they live in flocks and use vocalizations to interact with each other. Maybe talking to humans is a way of trying to fit in or communicate within a social group. If they're kept as pets, they might see humans as part of their social circle and try to mimic our sounds to bond or get attention.

Another angle is intelligence. Parrots are considered among the most intelligent birds. Their ability to mimic speech could be related to their problem-solving skills. Mimicking might be a form of learning, like how humans learn language. But is it just mimicry, or do they understand the wo

In [3]:
from langchain.tools import tool

@tool
def get_weather(location:str)-> str:
    """Get the weather at a location """
    return f"It's sunny is {location}"

model_with_tools = model.bind_tools([get_weather])

In [4]:
response = model_with_tools.invoke("What's the weather like in Boston")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking about the weather in Boston. Let me check the tools available. There\'s a function called get_weather that takes a location parameter. Since the user mentioned Boston, I need to call that function with "Boston" as the location. I should make sure the parameters are correctly formatted in JSON. Let me structure the tool call accordingly.\n', 'tool_calls': [{'id': '60kgxhxq6', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 96, 'prompt_tokens': 153, 'total_tokens': 249, 'completion_time': 0.150587608, 'completion_tokens_details': {'reasoning_tokens': 72}, 'prompt_time': 0.007485454, 'prompt_tokens_details': None, 'queue_time': 0.163691885, 'total_time': 0.158073062}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_2bfcc54d36', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': N

### Tool Execution Loops

In [5]:
# step 1: model generates tool calls
messages = [{
    "role": "user",
    "content": "What's the weather in Boston?"
}]

ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# step 2: execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# step 3: pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)

# "The current weather in Boston in 72 degree F and sunny"


The weather in Boston is sunny.


In [6]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Boston. Let me check the tools available. There\'s a function called get_weather that takes a location parameter. Since the user specified Boston, I need to call this function with "Boston" as the location. I\'ll make sure to format the tool call correctly within the XML tags as instructed.\n', 'tool_calls': [{'id': 'g4hqgx40t', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 93, 'prompt_tokens': 153, 'total_tokens': 246, 'completion_time': 0.13904036, 'completion_tokens_details': {'reasoning_tokens': 69}, 'prompt_time': 0.009665215, 'prompt_tokens_details': None, 'queue_time': 1.13901569, 'total_time': 0.148705575}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_d58dbe76cd', 'service_tier': 'on_demand'